<a href="https://colab.research.google.com/github/haroldrueda23-stack/Harold-Rueda-r/blob/main/Redes_Neuronales_V_Siamesa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Redes Neuronales V: Reconocimiento facial con Redes Siamesas y Triplet Loss

**Autor:** Harold Steven Rueda Lara
**Programa:** Administración de Empresas, Universidad de Cundinamarca, Extensión Facatativá
**Framework:** PyTorch (Google Colab)

**Objetivo:** implementar una red neuronal siamesa que aprenda *embeddings* faciales de 128 dimensiones mediante **Triplet Loss**, y evaluarla en verificación de identidad y en escenarios de **One-shot**, **Few-shot** y **Zero-shot Learning**.

**Nota ética:** todas las imágenes del dataset son propias o de personas que autorizaron explícitamente el uso de sus fotografías para fines académicos.

**Estructura de datos esperada:** una carpeta por persona, con varias fotos cada una (idealmente 8 o más por persona y al menos 4 personas):

```
dataset_rostros/
    persona_1/foto1.jpg, foto2.jpg, ...
    persona_2/...
```

## 1. Instalación e importaciones

In [ ]:
# facenet-pytorch se instala sin dependencias para no alterar la versión de torch de Colab
!pip -q install --no-deps facenet-pytorch

In [ ]:
import os, random, zipfile, shutil, glob
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from facenet_pytorch import MTCNN
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, accuracy_score, ConfusionMatrixDisplay
from sklearn.decomposition import PCA

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Dispositivo:', device)

## 2. Carga del dataset
Elige **una** opción: Google Drive (carpeta `dataset_rostros` en *Mi unidad*) o subir un `.zip` con la misma estructura.

In [ ]:
USE_DRIVE = True   # False para subir un .zip manualmente

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RAW_DIR = '/content/drive/MyDrive/dataset_rostros'
else:
    from google.colab import files
    up = files.upload()
    zname = list(up.keys())[0]
    with zipfile.ZipFile(zname) as z:
        z.extractall('/content/dataset_rostros')
    RAW_DIR = '/content/dataset_rostros'

personas = sorted([d for d in os.listdir(RAW_DIR) if os.path.isdir(os.path.join(RAW_DIR, d))])
resumen = {p: len([f for f in os.listdir(os.path.join(RAW_DIR, p)) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]) for p in personas}
print(pd.Series(resumen, name='imágenes').to_frame())

## 3. Preprocesamiento facial
Pasos aplicados a cada fotografía:
1. Corrección de orientación EXIF y conversión a RGB.
2. **Detección de rostro con MTCNN** y recorte con margen de 20 px.
3. Redimensionado a **160 x 160** píxeles.
4. Si no se detecta rostro, la imagen se descarta y se reporta (control de calidad del dataset).
5. En entrenamiento: aumento de datos (volteo horizontal, variación de brillo/contraste, pequeñas rotaciones y traslaciones). Normalización con media y desviación de ImageNet.

In [ ]:
FACES_DIR = '/content/faces'
shutil.rmtree(FACES_DIR, ignore_errors=True)
mtcnn = MTCNN(image_size=160, margin=20, post_process=False, select_largest=True, device=device)

descartadas = []
faces = defaultdict(list)   # identidad -> lista de rutas de rostros recortados

for p in personas:
    os.makedirs(os.path.join(FACES_DIR, p), exist_ok=True)
    for fn in sorted(os.listdir(os.path.join(RAW_DIR, p))):
        if not fn.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue
        ruta = os.path.join(RAW_DIR, p, fn)
        try:
            img = ImageOps.exif_transpose(Image.open(ruta)).convert('RGB')
            cara = mtcnn(img)
        except Exception as e:
            cara = None
        if cara is None:
            descartadas.append(ruta)
            continue
        arr = cara.permute(1, 2, 0).clamp(0, 255).byte().cpu().numpy()
        destino = os.path.join(FACES_DIR, p, os.path.splitext(fn)[0] + '.jpg')
        Image.fromarray(arr).save(destino, quality=95)
        faces[p].append(destino)

print('Rostros procesados por persona:', {k: len(v) for k, v in faces.items()})
print('Imágenes sin rostro detectado:', len(descartadas))
for d in descartadas: print('  -', d)

In [ ]:
# Evidencia visual del preprocesamiento: original vs rostro recortado
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
muestras = [(p, faces[p][0]) for p in list(faces)[:4]]
for i, (p, ruta) in enumerate(muestras):
    nombre = os.path.basename(ruta).rsplit('.', 1)[0]
    orig = [f for f in glob.glob(os.path.join(RAW_DIR, p, nombre + '.*'))][0]
    axes[0, i].imshow(ImageOps.exif_transpose(Image.open(orig)).convert('RGB')); axes[0, i].set_title(f'{p} (original)'); axes[0, i].axis('off')
    axes[1, i].imshow(Image.open(ruta)); axes[1, i].set_title('recorte 160x160'); axes[1, i].axis('off')
plt.tight_layout(); plt.show()

## 4. Partición de datos
- **Identidades vistas:** se usan para entrenar. Cada una se divide en imágenes de entrenamiento (75 %) y de prueba (25 %).
- **Identidades no vistas (~25 % de las personas):** nunca se usan en el entrenamiento; sirven para evaluar **Zero-shot** (la red debe comparar rostros de personas que jamás vio).

In [ ]:
ids = sorted([p for p in faces if len(faces[p]) >= 4])
assert len(ids) >= 4, 'Se necesitan al menos 4 personas con 4 o más fotos válidas'
rng = random.Random(SEED); rng.shuffle(ids)
n_unseen = max(1, round(0.25 * len(ids)))
unseen_ids = sorted(ids[:n_unseen])
train_ids = sorted(ids[n_unseen:])

train_imgs, test_imgs = {}, {}
for p in train_ids:
    lst = faces[p][:]; rng.shuffle(lst)
    k = max(2, int(round(0.75 * len(lst))))
    train_imgs[p], test_imgs[p] = lst[:k], lst[k:]
unseen_imgs = {p: faces[p] for p in unseen_ids}

print('Identidades de entrenamiento:', train_ids)
print('Identidades no vistas (zero-shot):', unseen_ids)
print('Imágenes train:', sum(len(v) for v in train_imgs.values()), '| test (vistas):', sum(len(v) for v in test_imgs.values()), '| no vistas:', sum(len(v) for v in unseen_imgs.values()))

## 5. Dataset de tripletas y transformaciones
Cada tripleta está formada por **ancla (A)**, **positivo (P)** (otra foto de la misma persona) y **negativo (N)** (foto de otra persona). Se muestrean aleatoriamente en cada época.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
tf_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
tf_eval = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class TripletFaces(Dataset):
    def __init__(self, id2imgs, transform, mult=6):
        self.id2imgs = {k: v for k, v in id2imgs.items() if len(v) >= 2}
        self.ids = list(self.id2imgs)
        self.anchors = [(k, p) for k, v in self.id2imgs.items() for p in v]
        self.transform, self.mult = transform, mult
    def __len__(self):
        return len(self.anchors) * self.mult
    def _load(self, path):
        return self.transform(Image.open(path).convert('RGB'))
    def __getitem__(self, i):
        idn, a = self.anchors[i % len(self.anchors)]
        p = random.choice([x for x in self.id2imgs[idn] if x != a])
        neg_id = random.choice([x for x in self.ids if x != idn])
        n = random.choice(self.id2imgs[neg_id])
        return self._load(a), self._load(p), self._load(n)

train_ds = TripletFaces(train_imgs, tf_train)
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=2, drop_last=True)
print('Tripletas por época:', len(train_ds))

## 6. Arquitectura de la red siamesa
Una red siamesa usa **la misma red con pesos compartidos** para procesar las tres imágenes de la tripleta. Aquí el *backbone* es una **ResNet-18 preentrenada en ImageNet** (transfer learning) seguida de una cabeza densa que proyecta a un **embedding de 128 dimensiones normalizado L2**. Así, la distancia euclídea entre dos embeddings mide la disimilitud entre rostros.

In [ ]:
class SiameseEmbedder(nn.Module):
    def __init__(self, dim=128):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        base.fc = nn.Identity()
        self.backbone = base
        self.head = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, dim))
    def forward(self, x):
        return F.normalize(self.head(self.backbone(x)), p=2, dim=1)

model = SiameseEmbedder().to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, 'M de parámetros')

## 7. Triplet Loss y entrenamiento
$$\mathcal{L}(A,P,N)=\max\big(\lVert f(A)-f(P)\rVert_2^2-\lVert f(A)-f(N)\rVert_2^2+\alpha,\;0\big)$$

donde $f$ es el embedding y $\alpha$ es el margen. La pérdida obliga a que el positivo quede más cerca del ancla que el negativo por al menos $\alpha$. En PyTorch se usa `nn.TripletMarginLoss` (distancia euclídea, margen 0.5).

In [ ]:
EPOCHS = 15
criterion = nn.TripletMarginLoss(margin=0.5, p=2)
optimizer = torch.optim.Adam([
    {'params': model.backbone.parameters(), 'lr': 1e-4},
    {'params': model.head.parameters(), 'lr': 1e-3}])
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

hist = {'loss': [], 'triplet_acc': []}
for ep in range(1, EPOCHS + 1):
    model.train(); tot, ok, n = 0.0, 0, 0
    for a, p, ng in train_dl:
        a, p, ng = a.to(device), p.to(device), ng.to(device)
        ea, ep_, en = model(a), model(p), model(ng)
        loss = criterion(ea, ep_, en)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        tot += loss.item() * a.size(0)
        ok += (F.pairwise_distance(ea, ep_) < F.pairwise_distance(ea, en)).sum().item(); n += a.size(0)
    scheduler.step()
    hist['loss'].append(tot / n); hist['triplet_acc'].append(ok / n)
    print(f'Época {ep:02d}/{EPOCHS} | loss {tot/n:.4f} | tripletas bien ordenadas {ok/n:.3f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(hist['loss']); ax[0].set_title('Triplet Loss (entrenamiento)'); ax[0].set_xlabel('Época')
ax[1].plot(hist['triplet_acc']); ax[1].set_title('Proporción d(A,P) < d(A,N)'); ax[1].set_xlabel('Época')
plt.tight_layout(); plt.show()

## 8. Verificación facial (¿son la misma persona?)
Se generan pares positivos (misma persona) y negativos (personas distintas). Se calcula la distancia entre embeddings y se decide con un **umbral**. El umbral se elige con las identidades **vistas (imágenes de prueba)** y se aplica sin cambios a las **identidades no vistas**.

In [ ]:
@torch.no_grad()
def embed(paths):
    model.eval(); out = []
    for i in range(0, len(paths), 32):
        x = torch.stack([tf_eval(Image.open(p).convert('RGB')) for p in paths[i:i+32]]).to(device)
        out.append(model(x).cpu())
    return torch.cat(out).numpy()

def make_pairs(id2imgs, max_pos=300, seed=SEED):
    r = random.Random(seed); pos, neg = [], []
    keys = list(id2imgs)
    for k in keys:
        L = id2imgs[k]
        pos += [(L[i], L[j]) for i in range(len(L)) for j in range(i + 1, len(L))]
    r.shuffle(pos); pos = pos[:max_pos]
    while len(neg) < len(pos):
        a, b = r.sample(keys, 2)
        neg.append((r.choice(id2imgs[a]), r.choice(id2imgs[b])))
    pairs = pos + neg; y = np.array([1] * len(pos) + [0] * len(neg))
    return pairs, y

def pair_distances(pairs):
    cache = {}
    paths = sorted({p for pr in pairs for p in pr})
    E = dict(zip(paths, embed(paths)))
    return np.array([np.linalg.norm(E[a] - E[b]) for a, b in pairs])

pairs_seen, y_seen = make_pairs(test_imgs)
d_seen = pair_distances(pairs_seen)

# Umbral óptimo en identidades vistas (imágenes de prueba)
ths = np.linspace(0, 2, 401)
accs = [accuracy_score(y_seen, (d_seen < t).astype(int)) for t in ths]
THRESH = float(ths[int(np.argmax(accs))])
print(f'Umbral elegido: {THRESH:.3f}')

def evaluar(nombre, pairs, y):
    d = pair_distances(pairs); pred = (d < THRESH).astype(int)
    acc = accuracy_score(y, pred); auc = roc_auc_score(y, -d)
    print(f'{nombre}: pares={len(y)} | exactitud={acc:.3f} | AUC={auc:.3f}')
    return d, pred, acc, auc

res = {}
res['vistas'] = evaluar('Identidades vistas (test)', pairs_seen, y_seen)
pairs_un, y_un = make_pairs(unseen_imgs)
res['no_vistas'] = evaluar('Identidades NO vistas (zero-shot)', pairs_un, y_un)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
for nombre, y, clave in [('Vistas', y_seen, 'vistas'), ('No vistas', y_un, 'no_vistas')]:
    d = res[clave][0]; fpr, tpr, _ = roc_curve(y, -d)
    ax[0].plot(fpr, tpr, label=f'{nombre} (AUC={res[clave][3]:.2f})')
ax[0].plot([0, 1], [0, 1], 'k--'); ax[0].set_title('Curva ROC'); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].legend()
d_s = res['vistas'][0]
ax[1].hist(d_s[y_seen == 1], bins=20, alpha=.6, label='misma persona'); ax[1].hist(d_s[y_seen == 0], bins=20, alpha=.6, label='distinta persona')
ax[1].axvline(THRESH, color='k', ls='--', label='umbral'); ax[1].set_title('Distancias (vistas)'); ax[1].legend()
ConfusionMatrixDisplay(confusion_matrix(y_un, res['no_vistas'][1]), display_labels=['distinta', 'misma']).plot(ax=ax[2], colorbar=False)
ax[2].set_title('Matriz de confusión (no vistas)')
plt.tight_layout(); plt.show()

## 9. One-shot, Few-shot y Zero-shot
- **One-shot:** 1 imagen de referencia por persona.
- **Few-shot:** *k* imágenes de referencia por persona (aquí k = 3 y 5); el prototipo es el promedio de sus embeddings.
- **Zero-shot (en el sentido de identidades):** las personas evaluadas **nunca participaron en el entrenamiento**; la red solo usa lo aprendido sobre comparación de rostros.

Cada imagen de consulta se clasifica con la referencia/prototipo más cercano. Se repite 30 veces con referencias aleatorias y se promedia.

In [ ]:
def nshot(id2imgs, k, reps=30, seed=SEED):
    r = random.Random(seed); accs = []
    keys = [x for x in id2imgs if len(id2imgs[x]) > k]
    if len(keys) < 2: return np.nan
    E = {x: dict(zip(id2imgs[x], embed(id2imgs[x]))) for x in keys}
    for _ in range(reps):
        protos, queries = {}, []
        for x in keys:
            sup = r.sample(id2imgs[x], k)
            protos[x] = np.mean([E[x][s] for s in sup], axis=0)
            queries += [(x, E[x][q]) for q in id2imgs[x] if q not in sup]
        P = np.stack([protos[x] for x in keys])
        ok = [keys[int(np.argmin(np.linalg.norm(P - e, axis=1)))] == x for x, e in queries]
        accs.append(np.mean(ok))
    return float(np.mean(accs))

filas = []
for nombre, d in [('Vistas (test)', test_imgs), ('No vistas (zero-shot)', unseen_imgs)]:
    d_ok = {x: v for x, v in d.items() if len(v) >= 2}
    for k, etiqueta in [(1, 'One-shot (k=1)'), (3, 'Few-shot (k=3)'), (5, 'Few-shot (k=5)')]:
        filas.append({'Conjunto': nombre, 'Escenario': etiqueta, 'Exactitud': nshot(d_ok, k)})
tabla = pd.DataFrame(filas).pivot(index='Escenario', columns='Conjunto', values='Exactitud').round(3)
tabla

## 10. Evidencias visuales: pares comparados y espacio de embeddings

In [ ]:
def mostrar_pares(pairs, y, d, n=6, titulo=''):
    idx = random.Random(1).sample(range(len(pairs)), min(n, len(pairs)))
    fig, axes = plt.subplots(2, len(idx), figsize=(3 * len(idx), 6.4))
    for c, i in enumerate(idx):
        a, b = pairs[i]; pred = d[i] < THRESH; ok = pred == bool(y[i])
        axes[0, c].imshow(Image.open(a)); axes[1, c].imshow(Image.open(b))
        axes[0, c].set_title(f'd={d[i]:.2f}\n{"MISMA" if pred else "DISTINTA"} ({"OK" if ok else "ERROR"})', color='green' if ok else 'red', fontsize=10)
        axes[0, c].axis('off'); axes[1, c].axis('off')
    fig.suptitle(titulo); plt.tight_layout(); plt.show()

mostrar_pares(pairs_seen, y_seen, res['vistas'][0], titulo='Pares de identidades vistas')
mostrar_pares(pairs_un, y_un, res['no_vistas'][0], titulo='Pares de identidades NO vistas')

In [ ]:
todas = {**{k: v for k, v in test_imgs.items()}, **unseen_imgs}
rutas, etiquetas = [], []
for k, v in todas.items():
    rutas += v; etiquetas += [k] * len(v)
Z = PCA(n_components=2, random_state=SEED).fit_transform(embed(rutas))
plt.figure(figsize=(7, 6))
for k in todas:
    m = np.array(etiquetas) == k
    plt.scatter(Z[m, 0], Z[m, 1], label=k + (' (no vista)' if k in unseen_ids else ''), marker='^' if k in unseen_ids else 'o', s=60)
plt.legend(fontsize=8); plt.title('Embeddings faciales (PCA 2D)'); plt.tight_layout(); plt.show()

## 11. Prueba de verificación con dos imágenes nuevas
Sube dos fotografías (de personas autorizadas) para verificar si pertenecen a la misma persona.

In [ ]:
@torch.no_grad()
def verificar(ruta1, ruta2):
    zs = []
    for r in (ruta1, ruta2):
        cara = mtcnn(ImageOps.exif_transpose(Image.open(r)).convert('RGB'))
        if cara is None: raise ValueError('No se detectó rostro en ' + r)
        arr = cara.permute(1, 2, 0).clamp(0, 255).byte().cpu().numpy()
        zs.append(model.eval()(tf_eval(Image.fromarray(arr)).unsqueeze(0).to(device)).cpu().numpy()[0])
    d = float(np.linalg.norm(zs[0] - zs[1]))
    print(f'Distancia = {d:.3f} | umbral = {THRESH:.3f} -> {"MISMA persona" if d < THRESH else "PERSONAS DISTINTAS"}')

# Ejemplo: verificar('/content/faces/persona_1/foto1.jpg', '/content/faces/persona_1/foto2.jpg')

In [ ]:
# Guardar modelo y resultados
os.makedirs('/content/salida', exist_ok=True)
torch.save({'state_dict': model.state_dict(), 'threshold': THRESH}, '/content/salida/siamesa_facial.pt')
tabla.to_csv('/content/salida/resultados_nshot.csv')
pd.DataFrame({'epoca': range(1, EPOCHS + 1), **hist}).to_csv('/content/salida/historial_entrenamiento.csv', index=False)
print('Archivos guardados en /content/salida')